# GeoAI Land Cover Classifier — 01 · Exploración de datos

**Dataset:** EuroSAT RGB (Sentinel-2) · **Entorno:** Google Colab · **Estado:** experimentos exploratorios.

**Ejecución:** `Entorno de ejecución → Ejecutar todo` desde una sesión nueva. Cada notebook prepara las variables que necesita; no depende de otro notebook abierto.

**Datos:** `./data` se descarga temporalmente en Colab. **Nunca utilices el conjunto test para escoger modelos.**

**Reproducibilidad:** `random_state=42`; pueden existir diferencias de tiempo o pequeñas variaciones entre versiones de dependencias.


In [ ]:

import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import PIL

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Pillow:", PIL.__version__)


In [ ]:

from torchvision.datasets import EuroSAT

dataset = EuroSAT(
    root="./data",
    download=True
)

print("Número de imágenes:", len(dataset))
print("Categorías:", dataset.classes)

imagen, etiqueta = dataset[0]

print("Tamaño de la imagen:", imagen.size)
print("Primera categoría:", dataset.classes[etiqueta])



## 02. Exploración visual del dataset

### Objetivo
Visualizar una imagen representativa de cada una de las
10 categorías de EuroSAT RGB.

### ¿Por qué es importante?
Antes de entrenar un modelo debemos conocer nuestros datos:

- Qué categorías existen.
- Qué diferencias visuales presentan.
- Qué clases podrían confundirse entre sí.
- Qué limitaciones tiene la resolución de 64 × 64 píxeles.

En esta etapa no entrenamos ningún modelo.
Realizamos un análisis exploratorio de datos (EDA).


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 02: Exploración visual de EuroSAT
# ============================================================

# Importamos pyplot para crear visualizaciones.
# Utilizamos el alias habitual "plt".
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Preparar la figura
# ------------------------------------------------------------

# Creamos una cuadrícula de 2 filas y 5 columnas.
# Así podremos mostrar las 10 categorías de EuroSAT.
figura, ejes = plt.subplots(
    nrows=2,
    ncols=5,
    figsize=(15, 7)
)

# Convertimos la matriz de ejes en una lista plana.
# Esto facilita recorrer todas las posiciones con un bucle.
ejes = ejes.flatten()

# ------------------------------------------------------------
# 2. Seleccionar una imagen de cada categoría
# ------------------------------------------------------------

# dataset.classes contiene los nombres de las 10 clases.
# dataset.samples contiene pares:
# (ruta_de_imagen, etiqueta_numerica)
#
# Seleccionamos la primera imagen encontrada de cada clase.
# Son ejemplos, no una muestra aleatoria ni representativa
# de toda la variabilidad del dataset.

for indice_clase, nombre_clase in enumerate(dataset.classes):

    # Buscamos el índice de la primera imagen cuya etiqueta
    # coincida con la clase que estamos recorriendo.
    indice_imagen = next(
        indice
        for indice, (_, etiqueta) in enumerate(dataset.samples)
        if etiqueta == indice_clase
    )

    # Recuperamos la imagen y su etiqueta.
    # La imagen es un objeto PIL.Image.
    imagen, etiqueta = dataset[indice_imagen]

    # Mostramos la imagen en su posición correspondiente.
    ejes[indice_clase].imshow(imagen)

    # Colocamos el nombre de la categoría como título.
    ejes[indice_clase].set_title(nombre_clase)

    # Ocultamos los ejes numéricos porque no aportan
    # información relevante a esta visualización.
    ejes[indice_clase].axis("off")

# ------------------------------------------------------------
# 3. Presentar el resultado
# ------------------------------------------------------------

# Añadimos un título general.
figura.suptitle(
    "EuroSAT RGB — Ejemplo de cada categoría",
    fontsize=16
)

# Ajustamos los espacios para evitar superposiciones.
plt.tight_layout()

# Mostramos la figura.
plt.show()



## 03. Distribución de categorías

### Objetivo
Analizar cuántas imágenes pertenecen a cada una de las
10 categorías de EuroSAT.

### Conceptos
- **Clase:** categoría que queremos predecir.
- **Etiqueta:** valor numérico que identifica una clase.
- **Distribución de clases:** cantidad de ejemplos por categoría.
- **Desbalance de clases:** situación en la que algunas clases
  tienen muchos más ejemplos que otras.

### ¿Por qué importa?
La distribución de las clases influye en el entrenamiento
y en la interpretación de las métricas del modelo.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 03: Distribución de categorías
# ============================================================

# Importamos Counter para contar cuántas veces
# aparece cada etiqueta en el dataset.
from collections import Counter

# Importamos pandas para crear una tabla.
import pandas as pd

# Importamos matplotlib para visualizar los resultados.
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Obtener las etiquetas
# ------------------------------------------------------------

# dataset.targets contiene la etiqueta numérica
# correspondiente a cada imagen.
#
# Por ejemplo:
# 0 = AnnualCrop
# 1 = Forest
# etc.
etiquetas = dataset.targets

# Contamos cuántas imágenes hay de cada categoría.
conteo = Counter(etiquetas)

# ------------------------------------------------------------
# 2. Crear una tabla de resultados
# ------------------------------------------------------------

# Construimos una lista de diccionarios.
# Cada diccionario representa una fila de la tabla.
filas = []

for indice_clase, nombre_clase in enumerate(dataset.classes):

    # Cantidad de imágenes de esta categoría.
    cantidad = conteo[indice_clase]

    # Porcentaje respecto al total del dataset.
    porcentaje = (cantidad / len(dataset)) * 100

    filas.append({
        "Categoría": nombre_clase,
        "Imágenes": cantidad,
        "Porcentaje": round(porcentaje, 2)
    })

# Convertimos la lista en un DataFrame de pandas.
distribucion = pd.DataFrame(filas)

# Mostramos la tabla ordenada de mayor a menor cantidad.
distribucion = distribucion.sort_values(
    by="Imágenes",
    ascending=False
)

display(distribucion)

# ------------------------------------------------------------
# 3. Representar las cantidades gráficamente
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.bar(
    distribucion["Categoría"],
    distribucion["Imágenes"]
)

plt.title("EuroSAT — Distribución de categorías")
plt.xlabel("Categoría")
plt.ylabel("Número de imágenes")

# Rotamos las etiquetas para facilitar su lectura.
plt.xticks(rotation=45, ha="right")

# Ajustamos los márgenes.
plt.tight_layout()

# Mostramos el gráfico.
plt.show()

# ------------------------------------------------------------
# 4. Comprobar el total
# ------------------------------------------------------------

print("Total de imágenes:", distribucion["Imágenes"].sum())
print("Número de categorías:", len(distribucion))


**Interpretación:** EuroSAT tiene 27.000 imágenes RGB de 64×64 distribuidas en diez clases. La distribución no es perfectamente uniforme. En el siguiente notebook se prepara la partición estratificada.